# Exercise 5.2
## Interpolation
In this task, we implement a simple NN to learn a complicated function.

In [1]:
import numpy as np
from tensorflow import keras
import matplotlib.pyplot as plt

layers = keras.layers
regularizers=keras.regularizers

### Generation of data

In [2]:
def some_complicated_function(x):
    return (
        (np.abs(x)) ** 0.5
        + 0.1 * x
        + 0.01 * x ** 2
        + 1
        - np.sin(x)
        + 0.5 * np.exp(x / 10.0)
        ) / (0.5 + np.abs(np.cos(x)))

Let's simulate the train data

In [3]:
N_train = 10 ** 4  # number of training samples
# Note: "[:, np.newaxis]" reshapes array to (N,1) as required by our DNN (we input one feature per sample)
xtrain = np.random.uniform(-10, 10, N_train)[:, np.newaxis]
ytrain = some_complicated_function(xtrain) + np.random.randn(xtrain.shape[0])  # train data includes some noise

In [4]:
print("xtrain.shape", xtrain.shape)
print("ytrain.shape", ytrain.shape)

xtrain.shape (10000, 1)
ytrain.shape (10000, 10000)


Simulate test data

In [5]:
N_test = 10000  # number of testing samples
xtest = np.linspace(-10, 10, N_test)
ytest = some_complicated_function(xtest)

In [6]:
print("xtest.shape", xtest.shape)
print("ytest.shape", ytest.shape)

xtest.shape (10000,)
ytest.shape (10000,)


### Define Model

Define the number of nodes, the number of layers, and choose an activation function.
Use `keras.regularizers` to use parameter norm penalties or add a dropout layer via `layers.Dropout(fraction)`.

You may use the skeleton below:

In [7]:
nb_nodes = 27
nb_layers = 3
#combinations tried: 
# (10,5)
#(20,5) with different parameters for L1 regularization (0.003,0.03,0.3)
#(30,5),(30,6),(28,4) with L2 regularization (0.03)
#(28,4) is better than (30,5) and (30,6)
#(28,4) with L1 regularization (0.03) #worst that with L2
#(28,4) with L1 0.03, L2 0.03
#(27,3) with L2 0.03
#(27,3) with L2 0.01
#(36,3) with L2 0.01
#(36,3) with L1 0.01 L2 0.01
#(36,4) with L1 0.01 L2 0.01
#(27,3) with L2 0.03 but with longer epoch (60)



#it seems that smaller number of layers -> higher stability
#high number of nodes <-> describe the complicated function

activation = "relu"
reg=regularizers.L2(0.03)

model = keras.models.Sequential(name="1Dfit")
model.add(layers.Dense(nb_nodes, activation=activation, input_dim=xtrain.shape[1],kernel_regularizer=reg))  # first layer

for n in range(nb_layers-1):
    model.add(layers.Dense(nb_nodes, activation=activation,kernel_regularizer=reg))
    

model.add(layers.Dense(1))  # final layer

print(model.summary())

Model: "1Dfit"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 27)                54        
                                                                 
 dense_1 (Dense)             (None, 27)                756       
                                                                 
 dense_2 (Dense)             (None, 27)                756       
                                                                 
 dense_3 (Dense)             (None, 1)                 28        
                                                                 
Total params: 1,594
Trainable params: 1,594
Non-trainable params: 0
_________________________________________________________________
None


### Compile the model (set an objective and choose an optimizer)

Choose an optimizer from `keras.optimizers`, e.g., `adam = keras.optimizers.Adam(lr=0.001)`.

Further, choose the correct objective (loss) for this <b>regression task</b>.

In [8]:
model.compile(loss="MeanSquaredError", optimizer=keras.optimizers.Adam(learning_rate=0.001))

### Train the model

Train the network for a couple of epochs and save the model several times in between.

In [9]:
epochs =  60
save_period = 10  # after how many epochs the model should be saved?

chkpnt_saver = keras.callbacks.ModelCheckpoint("weights-{epoch:02d}.hdf5", save_weights_only=True, save_freq=save_period)

results = model.fit(
    xtrain,
    ytrain,
    batch_size=64,
    epochs=epochs,
    verbose=1,
    callbacks=[chkpnt_saver]
    )

Epoch 1/60
157/157 [==============================] - 6s 34ms/step - loss: 15.7948
Epoch 2/60
157/157 [==============================] - 5s 31ms/step - loss: 12.5830
Epoch 3/60
157/157 [==============================] - 5s 29ms/step - loss: 16.0542
Epoch 4/60
157/157 [==============================] - 5s 32ms/step - loss: 18.7950
Epoch 5/60
157/157 [==============================] - 5s 29ms/step - loss: 26.3387
Epoch 6/60
157/157 [==============================] - 5s 30ms/step - loss: 27.1600
Epoch 7/60
157/157 [==============================] - 5s 32ms/step - loss: 37.4881
Epoch 8/60
157/157 [==============================] - 5s 29ms/step - loss: 33.4976
Epoch 9/60
157/157 [==============================] - 5s 30ms/step - loss: 60.7870
Epoch 10/60
157/157 [==============================] - 5s 29ms/step - loss: 42.9189
Epoch 11/60
157/157 [==============================] - 4s 29ms/step - loss: 46.0234
Epoch 12/60
157/157 [==============================] - 5s 29ms/step - loss: 63.6576
E

KeyboardInterrupt: 

Compare the performance of the model during the training. You may use the skeleton below:

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=2, figsize=(12, 8))

ax1.plot(xtest, ytest, color="black", label="data")
saved_epochs = range(save_period, epochs + 1, save_period)

colors = [plt.cm.jet((i + 1) / float(len(saved_epochs) + 1)) for i in range(len(saved_epochs))]

for i, epoch in enumerate(saved_epochs):
    model.load_weights("weights-{epoch:02d}.hdf5".format(epoch=epoch))
    ypredict = model.predict(xtest).squeeze()
    ax1.plot(xtest.squeeze(), ypredict, color=colors[i], label=epoch)
    ax2.plot(epoch, results.history["loss"][epoch - 1], color=colors[i], marker="o")

ax1.set(xlabel="x", ylabel="some_complicated_function(x)", xlim=(-10, 13), title="")
ax1.grid(True)
ax1.legend(loc="upper right", title="Epochs")

ax2.plot(results.history["loss"], color="black")
ax2.set(xlabel="epoch", ylabel="loss")
ax2.grid(True)
ax2.semilogy()

plt.show()